# March Madness LSTM Recurrent Model

This notebook builds an LSTM-based recurrent neural network to predict game outcomes using:
1. **Team Encoder**: LSTM to encode each team's historical stats to a latent vector
2. **Prediction Head**: Binary classifier combining two team embeddings
3. **Walk-Forward Training**: Temporal validation preventing future leakage

## Setup and Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Libraries imported")
print(f"  Device: {device}")

✓ Libraries imported
  Device: cuda


## Hyperparameters

In [2]:
# Model hyperparameters (easily adjustable)
CONFIG = {
    # Model architecture
    'latent_dim': 64,              # Dimensionality of team latent vector
    'hidden_dim': 128,             # LSTM hidden dimension
    'num_layers': 5,               # Number of LSTM layers
    'dropout': 0.3,                # Dropout rate
    'use_team_embedding': False,   # Include learnable team embeddings
    'team_embedding_dim': 16,      # Dimension of team embeddings (if used)
    
    # Training
    'batch_size': 32,
    'learning_rate': 1e-3,
    'epochs': 50,
}


## 1. Load and Prepare Data

In [3]:
# Load processed data
df = pd.read_csv("./InputData2023.csv")
df.columns

Index(['Team 1', 'Team 2', 'Date', 'Site', 'Outcome', '1-ORtg', '1-DRtg', '1-3PAr', '1-TS%', '1-TRB%', '1-AST%', '1-STL%', '1-BLK%', '1-eFG%', '1-TOV%', '1-ORB%', '1-FTr', '1-oeFG%', '1-oTOV%', '1-oDRB%', '1-oFTr', '1-Pace', '1-o3PAr', '2-ORtg', '2-DRtg', '2-3PAr', '2-TS%', '2-TRB%', '2-AST%', '2-STL%', '2-BLK%', '2-eFG%', '2-TOV%', '2-ORB%', '2-FTr', '2-oeFG%', '2-oTOV%', '2-oDRB%', '2-oFTr', '2-Pace', '2-o3PAr', 'Game ID'], dtype='object')

## 2. Team Encoder LSTM Model

In [4]:
class TeamEncoder(nn.Module):
    """Encodes a team's game history to a latent vector using LSTM."""
    
    def __init__(self, input_size, hidden_dim, num_layers, latent_dim, dropout=0.1):
        super(TeamEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.latent_dim = latent_dim
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Project LSTM output to latent space
        self.projection = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_length, input_size) - sequence of game stats
        Returns:
            latent: (batch_size, latent_dim) - team encoding
        """
        # LSTM forward pass
        _, (h_n, _) = self.lstm(x)  # h_n: (num_layers, batch_size, hidden_dim)
        
        # Take last layer's hidden state
        last_hidden = h_n[-1]  # (batch_size, hidden_dim)
        
        # Project to latent dimension
        latent = self.projection(last_hidden)  # (batch_size, latent_dim)
        
        return latent


class PredictionHead(nn.Module):
    """Binary classifier head that combines two team embeddings."""
    
    def __init__(self, latent_dim, hidden_dim=64):
        super(PredictionHead, self).__init__()
        
        # Take difference between teams as input
        self.fc1 = nn.Linear(latent_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, team_a_latent, team_b_latent):
        """
        Args:
            team_a_latent: (batch_size, latent_dim)
            team_b_latent: (batch_size, latent_dim)
        Returns:
            prob: (batch_size, 1) - probability team A wins
        """
        # Concatenate team embeddings
        combined = torch.cat([team_a_latent, team_b_latent], dim=1)
        
        # Forward pass
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        prob = self.sigmoid(x)
        
        return prob


class RecurrentGamePredictor(nn.Module):
    """Complete model: team encoders + prediction head."""
    
    def __init__(self, input_size, hidden_dim, num_layers, latent_dim, dropout=0.3):
        super(RecurrentGamePredictor, self).__init__()
        
        # Shared team encoder
        self.encoder = TeamEncoder(
            input_size=input_size,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            latent_dim=latent_dim,
            dropout=dropout
        )
        
        # Prediction head
        self.head = PredictionHead(latent_dim, hidden_dim=hidden_dim)
        
    def forward(self, team_a_seq, team_b_seq):
        """
        Args:
            team_a_seq: (batch_size, seq_length, input_size)
            team_b_seq: (batch_size, seq_length, input_size)
        Returns:
            prob: (batch_size, 1) - probability team A wins
        """
        # Encode teams
        team_a_latent = self.encoder(team_a_seq)
        team_b_latent = self.encoder(team_b_seq)
        
        # Predict
        prob = self.head(team_a_latent, team_b_latent)
        
        return prob, team_a_latent, team_b_latent


print("✓ Model architecture defined")
print(f"  TeamEncoder: LSTM({CONFIG['hidden_dim']}) -> Linear({CONFIG['latent_dim']})")
print(f"  PredictionHead: Concat(2×{CONFIG['latent_dim']}) -> Binary")

✓ Model architecture defined
  TeamEncoder: LSTM(128) -> Linear(64)
  PredictionHead: Concat(2×64) -> Binary


## 3. Data Preparation for Walk-Forward Training

In [5]:
## 3. Data Preparation and Feature Engineering

class GameSequenceDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for NCAA basketball games with lazy loading.
    
    For each game, returns:
    - team_1_seq: (n_prev_games1, n_features) - Team 1's ALL previous games
    - team_2_seq: (n_prev_games2, n_features) - Team 2's ALL previous games
    - outcome: 0 or 1 (Team 1 win?)
    
    Feature engineering happens at init to create model-ready inputs.
    Uses variable-length sequences (all available history).
    """
    
    def __init__(self, df):
        """
        Args:
            df: DataFrame with columns: Team 1, Team 2, Date, Site, Outcome, 
                1-ORtg, 1-DRtg, ... (stats prefixed with 1- and 2-)
        """
        self.df = df.reset_index(drop=True)
        
        # Auto-detect stat columns (all columns starting with '1-')
        self.stat_cols = sorted([col for col in df.columns if col.startswith('1-')])
        self.stat_cols_team2 = [col.replace('1-', '2-') for col in self.stat_cols]

        # Feature engineering at init
        self._prepare_features()
        
        # Build team history lookup
        self._build_team_history()
        


    def __len__(self):
        return len(self.df)
    
    
    def _prepare_features(self):
        """
        - Encode categorical variables (Team names, Site)
        - Normalize ranges
        """
        # 1. Encode Team names
        unique_teams = pd.concat([self.df['Team 1'], self.df['Team 2']]).unique()
        self.team_encoder = {team: idx for idx, team in enumerate(sorted(unique_teams))}
        self.n_teams = len(self.team_encoder)
        self.df['Team 1'] = self.df['Team 1'].map(self.team_encoder)
        self.df['Team 2'] = self.df['Team 2'].map(self.team_encoder)
        
        # 2. Encode Site/Location
        unique_sites = self.df['Site'].unique()
        self.site_encoder = {site: idx for idx, site in enumerate(sorted(unique_sites))}
        
        # 3. Encode Date features
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        # sort by date to ensure chronological order
        self.df = self.df.sort_values(by='Date').reset_index(drop=True)
        
        # 4. Normalization
        
    
    def _build_team_history(self):
        """Build lookup: team_name -> sorted list of game indices"""
        self.team_history = {}
        
        for idx, row in self.df.iterrows():
            team_1 = row['Team 1']
            team_2 = row['Team 2']
            
            if team_1 not in self.team_history:
                self.team_history[team_1] = []
            if team_2 not in self.team_history:
                self.team_history[team_2] = []
            
            self.team_history[team_1].append(idx)
            self.team_history[team_2].append(idx)
    

    def _get_team_sequence(self, team_name, current_game_idx):
        """
        Get ALL previous games for a team before current_game_idx.
        
        Args:
            team_name: Team identifier
            current_game_idx: Index of current game (exclude this and later)
        
        Returns:
            np.array: (n_prev_games, n_features) - sequence of all historical stats
                      Returns zeros if no previous games available
        """
        if team_name not in self.team_history:
            return np.zeros((0, len(self.stat_cols)), dtype=np.float32)
        
        # Get all previous games
        team_game_indices = [idx for idx in self.team_history[team_name] 
                            if idx < current_game_idx]
        
        if len(team_game_indices) == 0:
            return np.zeros((0, len(self.stat_cols)), dtype=np.float32)
        
        # Extract stats from ALL previous games (in chronological order)
        game_seqs = []
        for game_idx in team_game_indices:
            game_row = self.df.loc[game_idx]
            
            # Determine which team slot this team was in
            if game_row['Team 1'] == team_name:
                stats = game_row[self.stat_cols].values.astype(np.float32)
            else:
                stats = game_row[self.stat_cols_team2].values.astype(np.float32)
            
            game_seqs.append(stats)
        
        game_seqs = np.array(game_seqs, dtype=np.float32)
        
        return game_seqs
    

    def __getitem__(self, idx):
        """
        Returns:
            (team_1_seq, team_2_seq, outcome)
            - team_1_seq: (n_prev_games_1, n_features) - ALL Team 1 history
            - team_2_seq: (n_prev_games_2, n_features) - ALL Team 2 history
            - outcome: 0 or 1
            
        Note: Sequences are variable length! Use collate_fn for batching.
        """
        row = self.df.loc[idx]
        
        team_1 = row['Team 1']
        team_2 = row['Team 2']
        outcome = int(row['Outcome'])
        
        # Get ALL historical sequences (all games before this one)
        team_1_seq = self._get_team_sequence(team_1, idx)
        team_2_seq = self._get_team_sequence(team_2, idx)
        
        return (
            torch.FloatTensor(team_1_seq),
            torch.FloatTensor(team_2_seq),
            torch.LongTensor([outcome])
        )


# Custom collate function for variable-length sequences
def collate_variable_length(batch):
    """
    Collate variable-length sequences.
    Pads shorter sequences to match the longest in the batch.
    """
    team_a_seqs, team_b_seqs, outcomes = zip(*batch)
    # Find max lengths in this batch
    max_len_a = max(seq.shape[0] for seq in team_a_seqs) 
    max_len_b = max(seq.shape[0] for seq in team_b_seqs)
    
    n_features = team_a_seqs[0].shape[1]
    batch_size = len(batch)
    
    # Pad sequences
    team_a_padded = torch.zeros(batch_size, max_len_a, n_features)
    team_b_padded = torch.zeros(batch_size, max_len_b, n_features)
    for i, (seq_a, seq_b) in enumerate(zip(team_a_seqs, team_b_seqs)):
        if seq_a.shape[0] > 0:
            team_a_padded[i, :seq_a.shape[0]] = seq_a
        if seq_b.shape[0] > 0:
            team_b_padded[i, :seq_b.shape[0]] = seq_b
    
    outcomes_stacked = torch.cat(outcomes, dim=0)
    
    return team_a_padded, team_b_padded, outcomes_stacked


# Initialize dataset
print("Creating dataset with feature engineering...\n")
dataset = GameSequenceDataset(df)

# Test a sample
print(f"\n✓ Dataset ready!")
print(f"\nSample games (showing variable sequence lengths):")
for i in [100, 500, 1000]:
    team_1_seq, team_2_seq, outcome = dataset[i]
    print(f"\n  Game {i}:")
    print(f"    Team 1 history: {team_1_seq.shape[0]} previous games, stats shape {team_1_seq.shape}")
    print(f"    Team 2 history: {team_2_seq.shape[0]} previous games, stats shape {team_2_seq.shape}")
    print(f"    Outcome: {outcome.item()}")
    if team_1_seq.shape[0] > 0:
        print(f"    Team 1 most recent stats: {team_1_seq[-1, :5]}")
    if team_2_seq.shape[0] > 0:
        print(f"    Team 2 most recent stats: {team_2_seq[-1, :5]}")

Creating dataset with feature engineering...


✓ Dataset ready!

Sample games (showing variable sequence lengths):

  Game 100:
    Team 1 history: 1 previous games, stats shape torch.Size([1, 18])
    Team 2 history: 0 previous games, stats shape torch.Size([0, 18])
    Outcome: 0
    Team 1 most recent stats: tensor([3.7500e-01, 3.6842e-01, 4.2553e-02, 1.0422e+02, 4.1071e-01])

  Game 500:
    Team 1 history: 1 previous games, stats shape torch.Size([1, 18])
    Team 2 history: 3 previous games, stats shape torch.Size([3, 18])
    Outcome: 1
    Team 1 most recent stats: tensor([ 0.3836,  0.6667,  0.0879, 82.6261,  0.2808])
    Team 2 most recent stats: tensor([3.0994e-01, 4.2466e-01, 6.4815e-02, 1.1522e+02, 3.0409e-01])

  Game 1000:
    Team 1 history: 4 previous games, stats shape torch.Size([4, 18])
    Team 2 history: 5 previous games, stats shape torch.Size([5, 18])
    Outcome: 1
    Team 1 most recent stats: tensor([3.9827e-01, 4.8624e-01, 4.9587e-02, 1.0007e+02, 2.4675e-01])

## 5. Walk-Forward Training

In [6]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for team_a_seq, team_b_seq, targets in train_loader:
        team_a_seq = team_a_seq.to(device)
        team_b_seq = team_b_seq.to(device)
        targets = targets.to(device).view(-1, 1)
        
        optimizer.zero_grad()
        
        # Forward
        prob, _, _ = model(team_a_seq, team_b_seq)
        loss = criterion(prob, targets.float())
        
        # Backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def evaluate(model, val_loader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for team_a_seq, team_b_seq, targets in val_loader:
            team_a_seq = team_a_seq.to(device)
            team_b_seq = team_b_seq.to(device)
            targets = targets.to(device).view(-1, 1)
            
            prob, _, _ = model(team_a_seq, team_b_seq)
            loss = criterion(prob, targets.float())
            
            total_loss += loss.item()
            all_probs.append(prob.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    
    return total_loss / len(val_loader), all_probs, all_targets


print("✓ Training functions defined")

✓ Training functions defined


## 6. Execute Walk-Forward Validation

In [7]:
model = RecurrentGamePredictor(
    input_size=len(dataset.stat_cols),
    hidden_dim=CONFIG['hidden_dim'],
    num_layers=CONFIG['num_layers'],
    latent_dim=CONFIG['latent_dim'],
    dropout=CONFIG['dropout']
).to(device)

# Setup optimizer and loss
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)
criterion = nn.BCELoss()

# Create DataLoader
train_loader = DataLoader(
    dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True,
    collate_fn=collate_variable_length
)

for epoch in tqdm(range(CONFIG['epochs'])):
    model.train()
    total_loss = 0
    
    for team_a_seq, team_b_seq, targets in train_loader:
        # Move to device
        team_a_seq = team_a_seq.to(device)
        team_b_seq = team_b_seq.to(device)
        targets = targets.to(device).view(-1, 1).float()
        
        # Forward pass
        optimizer.zero_grad()
        prob, _, _ = model(team_a_seq, team_b_seq)
        loss = criterion(prob, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
         
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {avg_loss:.4f}")

print("\n✓ Training completed!")

 20%|█████████████████▊                                                                       | 10/50 [10:10<40:28, 60.71s/it]

  Epoch 10/50 - Loss: 0.6924


 40%|███████████████████████████████████▌                                                     | 20/50 [20:03<29:32, 59.09s/it]

  Epoch 20/50 - Loss: 0.6920


 60%|█████████████████████████████████████████████████████▍                                   | 30/50 [29:51<19:37, 58.86s/it]

  Epoch 30/50 - Loss: 0.6918


 80%|███████████████████████████████████████████████████████████████████████▏                 | 40/50 [39:39<09:48, 58.89s/it]

  Epoch 40/50 - Loss: 0.6915


100%|█████████████████████████████████████████████████████████████████████████████████████████| 50/50 [49:30<00:00, 59.41s/it]

  Epoch 50/50 - Loss: 0.6919

✓ Training completed!


In [8]:
print("="*70)
print("WALK-FORWARD VALIDATION")
print("="*70)

# Store results
all_results = {
    'window': [],
    'train_loss': [],
    'val_loss': [],
    'test_accuracy': [],
    'test_auc': [],
    'all_test_probs': [],
    'all_test_targets': []
}

# Walk-forward: train on all data up to season N, test on season N+1
for i, test_season in enumerate(seasons[:-1]):
    train_seasons = seasons[:i+1]
    
    print(f"\nWindow {i+1}: Train [{train_seasons[0]}-{train_seasons[-1]}] -> Test [{test_season}]")
    
    # Prepare training data (80% of data)
    train_matchups = []
    for s in train_seasons:
        n = len(season_data[s])
        split_idx = int(n * 0.8)
        train_matchups.extend(season_data[s][:split_idx])
    
    # Prepare validation data (20% of data)
    val_matchups = []
    for s in train_seasons:
        n = len(season_data[s])
        split_idx = int(n * 0.8)
        val_matchups.extend(season_data[s][split_idx:])
    
    # Test data: all of next season
    test_matchups = season_data[test_season]
    
    print(f"  Train: {len(train_matchups):,} | Val: {len(val_matchups):,} | Test: {len(test_matchups):,}")
    
    # Create tensors
    def matchup_to_tensor(matchup_list):
        team_a_seqs = np.array([data['team_a_seq'] for _, data in matchup_list])
        team_b_seqs = np.array([data['team_b_seq'] for _, data in matchup_list])
        targets = np.array([data['target'] for _, data in matchup_list])
        
        team_a_seqs = torch.FloatTensor(team_a_seqs)
        team_b_seqs = torch.FloatTensor(team_b_seqs)
        targets = torch.LongTensor(targets)
        
        return TensorDataset(team_a_seqs, team_b_seqs, targets)
    
    train_dataset = matchup_to_tensor(train_matchups)
    val_dataset = matchup_to_tensor(val_matchups)
    test_dataset = matchup_to_tensor(test_matchups)
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # Initialize model
    model = RecurrentGamePredictor(
        input_size=len(feature_cols),
        hidden_dim=CONFIG['hidden_dim'],
        num_layers=CONFIG['num_layers'],
        latent_dim=CONFIG['latent_dim'],
        dropout=CONFIG['dropout']
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    criterion = nn.BCELoss()
    
    # Train
    best_val_loss = float('inf')
    patience_counter = 0
    best_model = None
    
    for epoch in range(CONFIG['epochs']):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, _, _ = evaluate(model, val_loader, criterion, device)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= CONFIG['patience']:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model)
    
    # Evaluate on test set
    test_loss, test_probs, test_targets = evaluate(model, test_loader, criterion, device)
    test_preds = (test_probs > 0.5).astype(int).flatten()
    test_accuracy = accuracy_score(test_targets, test_preds)
    test_auc = roc_auc_score(test_targets, test_probs)
    
    print(f"  Test Accuracy: {test_accuracy:.4f} | Test AUC: {test_auc:.4f}")
    
    all_results['window'].append(i+1)
    all_results['train_loss'].append(train_loss)
    all_results['val_loss'].append(best_val_loss)
    all_results['test_accuracy'].append(test_accuracy)
    all_results['test_auc'].append(test_auc)
    all_results['all_test_probs'].append(test_probs)
    all_results['all_test_targets'].append(test_targets)

print("\n" + "="*70)
print("✓ Walk-forward validation completed!")

WALK-FORWARD VALIDATION


NameError: name 'seasons' is not defined

## 7. Results Summary

In [ ]:
print("\n" + "="*70)
print("WALK-FORWARD RESULTS")
print("="*70)

results_df = pd.DataFrame(all_results)
print("\nPer-Window Performance:")
print(results_df[['window', 'train_loss', 'val_loss', 'test_accuracy', 'test_auc']].to_string(index=False))

print(f"\n📊 Overall Statistics:")
print(f"  Mean Test Accuracy: {np.mean(all_results['test_accuracy']):.4f} ± {np.std(all_results['test_accuracy']):.4f}")
print(f"  Mean Test AUC:      {np.mean(all_results['test_auc']):.4f} ± {np.std(all_results['test_auc']):.4f}")
print(f"  Min Test Accuracy:  {np.min(all_results['test_accuracy']):.4f}")
print(f"  Max Test Accuracy:  {np.max(all_results['test_accuracy']):.4f}")

# Combined predictions across all windows
all_probs = np.concatenate(all_results['all_test_probs'])
all_targets = np.concatenate(all_results['all_test_targets'])
all_preds = (all_probs > 0.5).astype(int).flatten()

print(f"\n📈 Aggregated Performance (All Test Sets):")
print(f"  Accuracy: {accuracy_score(all_targets, all_preds):.4f}")
print(f"  AUC-ROC:  {roc_auc_score(all_targets, all_probs):.4f}")
print(f"\n{classification_report(all_targets, all_preds, target_names=['Loss', 'Win'])}")

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy per window
axes[0, 0].plot(all_results['window'], all_results['test_accuracy'], marker='o', linewidth=2)
axes[0, 0].axhline(y=0.5, color='r', linestyle='--', label='Baseline (50%)')
axes[0, 0].set_xlabel('Window')
axes[0, 0].set_ylabel('Test Accuracy')
axes[0, 0].set_title('Accuracy per Walk-Forward Window')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

# AUC per window
axes[0, 1].plot(all_results['window'], all_results['test_auc'], marker='s', color='orange', linewidth=2)
axes[0, 1].axhline(y=0.5, color='r', linestyle='--', label='Baseline (50%)')
axes[0, 1].set_xlabel('Window')
axes[0, 1].set_ylabel('Test AUC-ROC')
axes[0, 1].set_title('AUC-ROC per Walk-Forward Window')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

# Loss curves
axes[1, 0].plot(all_results['window'], all_results['train_loss'], marker='o', label='Train', linewidth=2)
axes[1, 0].plot(all_results['window'], all_results['val_loss'], marker='s', label='Val', linewidth=2)
axes[1, 0].set_xlabel('Window')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Loss per Window')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

# ROC Curve (aggregated)
fpr, tpr, _ = roc_curve(all_targets, all_probs)
auc = roc_auc_score(all_targets, all_probs)
axes[1, 1].plot(fpr, tpr, lw=2, label=f'ROC (AUC={auc:.4f})')
axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve (All Test Sets)')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("✓ Visualizations complete")

## 9. Configuration Notes

**To adjust model performance, modify hyperparameters in cell 2:**

- `latent_dim`: Increase (64→128) for more complex team representations
- `hidden_dim`: LSTM capacity (128→256 for deeper modeling)
- `num_layers`: Stacking LSTMs (2→3 for more temporal depth)
- `sequence_length`: More history (10→20 games)
- `learning_rate`: Adjustment for convergence
- `batch_size`: Training dynamics
- `use_team_embedding`: Add learnable team identity vectors (not yet implemented)

**Walk-forward strategy:**
- Trains on all data up to season N (80% train, 20% val per season)
- Tests on season N+1
- Prevents temporal leakage
- Realistic evaluation of future season performance

# March Madness Season-based Model

This notebook builds a machine learning model to predict game outcomes **before** they happen using:
1. Accumulative season stats

1. Building our model:

Our model should work recurrently based on the temporal order of happened games. It is composed of the following modules:

a. A recurrent network that is able to encoder one team's up-to-date stats to a latent vector.

b. A prediction head that uses two embeddings of the team and can predict which team wins/whether the underdog will win.

How training works?
For each game, we collect all previous game stats of both teams as input and then use the score difference as output


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Libraries imported")
print(f"Working directory: {Path.cwd()}")

## 1. Load and Prepare Data

In [ ]:
# Load processed data
df = pd.read_csv("processed_data/ncaa_basketball_processed_2003_2023.csv")

# Convert dates and sort chronologically (CRITICAL for temporal features)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.sort_values(['season', 'game_date', 'game_id']).reset_index(drop=True)

print(f"✓ Data loaded: {df.shape}")
print(f"  Records: {len(df):,}")
print(f"  Seasons: {df['season'].min()}-{df['season'].max()}")
print(f"  Date range: {df['game_date'].min().date()} to {df['game_date'].max().date()}")

## 2. Feature Engineering - Rolling Statistics

Create pre-game features using only historical data (no future leakage).

In [ ]:
print("Creating historical features...")

# Define stats to track
stats = ['team_score', 'field_goal_pct', 'three_point_field_goal_pct', 
         'total_rebounds', 'assists', 'team_turnovers']
stats = [s for s in stats if s in df.columns]

# Calculate rolling averages (last 5 games, excluding current game)
for stat in stats:
    df[f'{stat}_L5'] = df.groupby(['team_id', 'season'])[stat].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

# Win rate (last 5 games)
df['win_rate_L5'] = df.groupby(['team_id', 'season'])['team_winner_encoded'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# Games played this season (before current game)
df['games_played'] = df.groupby(['team_id', 'season']).cumcount()

# Season win rate (before current game)
df['season_wins'] = df.groupby(['team_id', 'season'])['team_winner_encoded'].transform(
    lambda x: x.shift(1).expanding().sum()
).fillna(0)
df['season_win_rate'] = df.apply(
    lambda row: row['season_wins'] / row['games_played'] if row['games_played'] > 0 else 0.5,
    axis=1
)

print(f"✓ Created {len([c for c in df.columns if '_L5' in c or 'season_' in c])} features")
print(f"  Total columns: {len(df.columns)}")

## 3. Create Matchup Dataset

Transform from team-level records to matchup-level (Team A vs Team B).

In [ ]:
print("Creating matchup dataset...")

# Get complete games (both teams present)
game_counts = df.groupby('game_id').size()
complete_games = game_counts[game_counts == 2].index
df_complete = df[df['game_id'].isin(complete_games)].copy()

print(f"  Complete games: {len(complete_games):,}")

# Build matchup records
matchups = []
feature_cols = [c for c in df.columns if '_L5' in c or 'win_rate' in c or 'season_win' in c or 'games_played' in c]

for game_id in complete_games:
    game = df_complete[df_complete['game_id'] == game_id]
    if len(game) != 2:
        continue
    
    team_a, team_b = game.iloc[0], game.iloc[1]
    
    # Randomly shuffle which team is "Team A" to balance win/loss distribution
    if np.random.random() > 0.5:
        team_a, team_b = team_b, team_a
    
    matchup = {
        'game_id': game_id,
        'season': team_a['season'],
        'game_date': team_a['game_date'],
        'team_a_id': team_a['team_id'],
        'team_b_id': team_b['team_id'],
        'team_a_home': int(team_a.get('home_away_encoded', 0) == 1),
        'team_a_won': int(team_a.get('team_winner_encoded', 0))
    }
    
    # Add differential features (Team A - Team B)
    for col in feature_cols:
        if pd.notna(team_a[col]) and pd.notna(team_b[col]):
            matchup[f'diff_{col}'] = team_a[col] - team_b[col]
    
    matchups.append(matchup)

df_matchups = pd.DataFrame(matchups)
df_matchups['game_date'] = pd.to_datetime(df_matchups['game_date'])

print(f"✓ Matchups created: {len(df_matchups):,}")
print(f"  Features: {len([c for c in df_matchups.columns if 'diff_' in c])} differential features")

# Check class balance
print(f"\nClass distribution:")
win_count = (df_matchups['team_a_won'] == 1).sum()
loss_count = (df_matchups['team_a_won'] == 0).sum()
print(f"  Team A Wins:   {win_count:,} ({win_count/len(df_matchups)*100:.1f}%)")
print(f"  Team A Losses: {loss_count:,} ({loss_count/len(df_matchups)*100:.1f}%)")

print(f"\nSample:")
display(df_matchups[['game_id', 'season', 'team_a_id', 'team_b_id', 'team_a_home', 'team_a_won']].head())

## 4. Temporal Train/Test Split

For each season, use first 80% of games for training and last 20% for testing.

In [ ]:
print("Performing temporal split...")

# Clean data
df_clean = df_matchups.dropna(subset=['team_a_won']).copy()
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)

print(f"  Clean matchups: {len(df_clean):,}")

# Split by season: first 80% train, last 20% test
train_list = []
test_list = []

for season in df_clean['season'].unique():
    season_data = df_clean[df_clean['season'] == season].sort_values('game_date')
    n = len(season_data)
    split_idx = int(n * 0.8)
    
    train_list.append(season_data.iloc[:split_idx])
    test_list.append(season_data.iloc[split_idx:])

train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)

print(f"\n✓ Temporal split completed:")
print(f"  Train: {len(train_df):,} matchups ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} matchups ({len(test_df)/len(df_clean)*100:.1f}%)")
print(f"\n  Train date range: {train_df['game_date'].min().date()} to {train_df['game_date'].max().date()}")
print(f"  Test date range:  {test_df['game_date'].min().date()} to {test_df['game_date'].max().date()}")

## 5. Prepare Features and Target

In [ ]:
# Select features
exclude = ['game_id', 'season', 'game_date', 'team_a_id', 'team_b_id', 'team_a_won']
feature_cols = [c for c in df_clean.columns if c not in exclude]

X_train = train_df[feature_cols].copy()
y_train = train_df['team_a_won'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['team_a_won'].copy()

print(f"Features: {len(feature_cols)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

print(f"\nClass distribution:")
print(f"  Train - Win: {y_train.mean():.1%}, Loss: {(1-y_train.mean()):.1%}")
print(f"  Test  - Win: {y_test.mean():.1%}, Loss: {(1-y_test.mean()):.1%}")

## 6. Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled (StandardScaler)")

## 7. Train Logistic Regression Model

In [ ]:
print("Training logistic regression model...")

model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)

print(f"✓ Model trained (converged in {model.n_iter_[0]} iterations)")

## 8. Evaluate Model

In [ ]:
# Predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

print("="*70)
print("MODEL PERFORMANCE")
print("="*70)

print("\nTRAIN SET:")
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba):.4f}")

print("\nTEST SET:")
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba):.4f}")

print("\n" + classification_report(y_test, y_test_pred, target_names=['Loss', 'Win']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
axes[0].set_title('Train Confusion Matrix')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[0],
    'Abs_Coefficient': np.abs(model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 Most Important Features:")
print("="*70)
for idx, row in feature_importance.head(10).iterrows():
    direction = "↑" if row['Coefficient'] > 0 else "↓"
    print(f"  {row['Feature']:<40} {row['Coefficient']:>8.4f} {direction}")

# Visualize
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(12)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
plt.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Coefficient')
plt.title('Feature Importance (Top 12)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)
print(f"\n📊 Dataset: {len(df_clean):,} matchups")
print(f"   Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"\n🎯 Model: Logistic Regression")
print(f"   Features: {len(feature_cols)} (historical rolling stats)")
print(f"   Split: Temporal 80/20 per season")
print(f"\n📈 Performance:")
print(f"   Test Accuracy: {accuracy_score(y_test, y_test_pred):.2%}")
print(f"   Test ROC-AUC:  {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"\n🔑 Top Predictor: {feature_importance.iloc[0]['Feature']}")
print(f"\n💡 Next Steps:")
print(f"   • Add more historical windows (L10, L15)")
print(f"   • Try Random Forest / XGBoost")
print(f"   • Add head-to-head features")
print(f"   • Implement Elo ratings")
print("="*70)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Machine Learning imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")
print(f"Current working directory: {Path.cwd()}")

## 1. Load Processed Data

Load the cleaned and processed dataset from the previous data preparation notebook.

In [ ]:
# Load the processed data
data_path = Path("processed_data/ncaa_basketball_processed_2003_2023.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"✓ Data loaded successfully!")
    print(f"  File: {data_path}")
    print(f"  Shape: {df.shape}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {df.shape[1]}")
else:
    print(f"✗ ERROR: File not found at {data_path}")
    print(f"  Please run the data preparation notebook first to generate the processed data.")

In [ ]:
# Display basic information
print("\nDATASET OVERVIEW:")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Date range: {df['season'].min()} to {df['season'].max()}")
print(f"\nFirst few rows:")
display(df.head())

print(f"\nColumn names:")
print(df.columns.tolist())

## 2. Feature Selection and Data Preparation

Select features for modeling, handling missing values, and prepare the target variable.

In [ ]:
print("FEATURE SELECTION AND DATA PREPARATION")
print("="*70)

# Define target variable
target_col = 'team_winner_encoded' if 'team_winner_encoded' in df.columns else 'team_winner'

# Define columns to exclude from features
exclude_cols = [
    # Identifiers
    'game_id', 'team_id', 'opponent_id_combined',
    
    # Dates (use season instead)
    'game_date', 'game_date_time',
    
    # Target variables (and their raw versions)
    'team_winner', 'team_winner_encoded',
    
    # Location information (already have encoded version)
    'team_location', 'opponent_team_location',
    
    # Raw categorical features (use encoded versions)
    'season_type', 'home_away_combined',
    
    # Combined format columns (use individual made/attempted columns)
    'field_goals_made_field_goals_attempted',
    'three_point_field_goals_made_three_point_field_goals_attempted',
    'free_throws_made_free_throws_attempted',
]

# Get all columns
all_cols = df.columns.tolist()

# Select feature columns (exclude target and non-feature columns)
feature_cols = [col for col in all_cols if col not in exclude_cols]

print(f"Total columns in dataset: {len(all_cols)}")
print(f"Excluded columns: {len(exclude_cols)}")
print(f"Feature columns selected: {len(feature_cols)}")
print(f"Target variable: {target_col}")

In [ ]:
# Check for missing values in features and target
print("\nCHECKING MISSING VALUES:")
print("="*60)

# Check target variable
target_missing = df[target_col].isnull().sum()
print(f"\nTarget variable '{target_col}': {target_missing:,} missing ({target_missing/len(df)*100:.2f}%)")

# Check feature columns
print(f"\nFeature columns with missing values:")
missing_counts = df[feature_cols].isnull().sum()
missing_features = missing_counts[missing_counts > 0].sort_values(ascending=False)

if len(missing_features) > 0:
    print(f"Found {len(missing_features)} features with missing values:\n")
    for col, count in missing_features.items():
        pct = count / len(df) * 100
        print(f"  {col:<40}: {count:>8,} ({pct:>5.1f}%)")
else:
    print("  ✓ No missing values found in feature columns!")

In [ ]:
# Remove rows with missing target variable
print("\nCLEANING DATA:")
print("="*60)

initial_rows = len(df)
df_clean = df.dropna(subset=[target_col]).copy()
rows_removed = initial_rows - len(df_clean)

print(f"Rows with missing target removed: {rows_removed:,}")
print(f"Remaining rows: {len(df_clean):,}")

# For features, only keep rows where ALL features are non-null
df_clean = df_clean.dropna(subset=feature_cols)
final_rows = len(df_clean)
total_removed = initial_rows - final_rows

print(f"\nRows with any missing features removed: {initial_rows - final_rows:,}")
print(f"Final dataset size: {final_rows:,} rows ({final_rows/initial_rows*100:.1f}% of original)")

# Check class distribution
print(f"\nTARGET CLASS DISTRIBUTION:")
class_dist = df_clean[target_col].value_counts()
class_pct = df_clean[target_col].value_counts(normalize=True) * 100

for class_val in sorted(class_dist.index):
    label = "Loss" if class_val == 0 else "Win"
    print(f"  {label} ({class_val}): {class_dist[class_val]:,} ({class_pct[class_val]:.1f}%)")

# Check for class imbalance
imbalance_ratio = class_dist.max() / class_dist.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 1.5:
    print("  ⚠ Warning: Significant class imbalance detected")
else:
    print("  ✓ Classes are reasonably balanced")

## 3. Train/Test Split

Split the data into training and testing sets. We'll use 80% for training and 20% for testing.

In [ ]:
print("TRAIN/TEST SPLIT")
print("="*70)

# Prepare features (X) and target (y)
X = df_clean[feature_cols].copy()
y = df_clean[target_col].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y  # Maintain class distribution in both sets
)

print(f"\nSPLIT RESULTS:")
print(f"  Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test set:     {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

# Verify class distribution in splits
print(f"\nClass distribution in training set:")
train_dist = y_train.value_counts(normalize=True) * 100
for class_val in sorted(train_dist.index):
    label = "Loss" if class_val == 0 else "Win"
    print(f"  {label}: {train_dist[class_val]:.1f}%")

print(f"\nClass distribution in test set:")
test_dist = y_test.value_counts(normalize=True) * 100
for class_val in sorted(test_dist.index):
    label = "Loss" if class_val == 0 else "Win"
    print(f"  {label}: {test_dist[class_val]:.1f}%")

print("\n✓ Data split completed successfully!")

## 4. Feature Scaling

Scale features to have mean=0 and std=1 for better model performance.

In [ ]:
print("FEATURE SCALING")
print("="*70)

# Initialize scaler
scaler = StandardScaler()

# Fit on training data only (avoid data leakage)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Features scaled using StandardScaler")
print(f"  Training set scaled shape: {X_train_scaled.shape}")
print(f"  Test set scaled shape: {X_test_scaled.shape}")

# Show scaling statistics for a few features
print(f"\nScaling statistics (first 5 features):")
print(f"  Feature means: {scaler.mean_[:5].round(2)}")
print(f"  Feature stds:  {scaler.scale_[:5].round(2)}")

print("\n✓ Feature scaling completed!")

## 5. Train Logistic Regression Model

Train a baseline logistic regression model to predict wins/losses.

In [ ]:
print("TRAINING LOGISTIC REGRESSION MODEL")
print("="*70)

# Initialize model
model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced'  # Handle any slight class imbalance
)

print("Training model...")
print(f"  Model: Logistic Regression")
print(f"  Solver: lbfgs")
print(f"  Max iterations: 1000")
print(f"  Class weight: balanced")

# Train the model
model.fit(X_train_scaled, y_train)

print(f"\n✓ Model training completed!")
print(f"  Converged: {model.n_iter_[0] < 1000}")
print(f"  Iterations: {model.n_iter_[0]}")

## 6. Model Evaluation

Evaluate model performance on both training and test sets.

In [ ]:
print("MODEL EVALUATION")
print("="*70)

# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Predict probabilities for ROC-AUC
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
print("\nTRAINING SET PERFORMANCE:")
print("-"*40)
print(f"  Accuracy:  {accuracy_score(y_train, y_train_pred):.4f}")
print(f"  Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"  Recall:    {recall_score(y_train, y_train_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_train, y_train_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_train, y_train_proba):.4f}")

print("\nTEST SET PERFORMANCE:")
print("-"*40)
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba):.4f}")

# Check for overfitting
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
acc_diff = train_acc - test_acc

print(f"\nOVERFITTING CHECK:")
print("-"*40)
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")
print(f"  Difference:     {acc_diff:.4f}")

if acc_diff < 0.02:
    print("  ✓ Excellent generalization - minimal overfitting")
elif acc_diff < 0.05:
    print("  ✓ Good generalization - slight overfitting")
else:
    print("  ⚠ Moderate overfitting detected")

In [ ]:
# Detailed classification report
print("\nDETAILED CLASSIFICATION REPORT (Test Set):")
print("="*70)
print(classification_report(y_test, y_test_pred, target_names=['Loss (0)', 'Win (1)']))

In [ ]:
# Confusion Matrix Visualization
print("CONFUSION MATRIX")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set confusion matrix
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
axes[0].set_title('Training Set Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Test set confusion matrix
cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
axes[1].set_title('Test Set Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("\nTest Set Confusion Matrix Values:")
print(f"  True Negatives (Correct Loss predictions): {cm_test[0, 0]:,}")
print(f"  False Positives (Loss predicted as Win):   {cm_test[0, 1]:,}")
print(f"  False Negatives (Win predicted as Loss):   {cm_test[1, 0]:,}")
print(f"  True Positives (Correct Win predictions):  {cm_test[1, 1]:,}")

In [ ]:
# ROC Curve
print("\nROC CURVE")
print("="*70)

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✓ ROC-AUC Score: {roc_auc:.4f}")
if roc_auc > 0.9:
    print("  Excellent discrimination ability")
elif roc_auc > 0.8:
    print("  Good discrimination ability")
elif roc_auc > 0.7:
    print("  Acceptable discrimination ability")
else:
    print("  Poor discrimination ability")

## 7. Feature Importance Analysis

Analyze which features are most important for predicting wins/losses.

In [ ]:
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Get feature coefficients
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[0],
    'Abs_Coefficient': np.abs(model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("\nTop 20 Most Important Features:")
print("-"*70)
print(f"{'Rank':<6} {'Feature':<40} {'Coefficient':<12}")
print("-"*70)

for idx, (i, row) in enumerate(feature_importance.head(20).iterrows(), 1):
    direction = "↑" if row['Coefficient'] > 0 else "↓"
    print(f"{idx:<6} {row['Feature']:<40} {row['Coefficient']:>10.4f} {direction}")

print("\n↑ = Positive correlation with winning")
print("↓ = Negative correlation with winning")

In [ ]:
# Visualize top feature importances
plt.figure(figsize=(12, 8))

top_features = feature_importance.head(15)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]

plt.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Top 15 Feature Importances (Logistic Regression Coefficients)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Feature importance analysis completed!")

## 8. Cross-Validation

Perform k-fold cross-validation to ensure model robustness.

In [ ]:
print("CROSS-VALIDATION")
print("="*70)

# Perform 5-fold cross-validation
print("Performing 5-fold cross-validation...")
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f"\nCross-Validation Results:")
print("-"*40)
print(f"  Fold scores: {[f'{score:.4f}' for score in cv_scores]}")
print(f"  Mean accuracy: {cv_scores.mean():.4f}")
print(f"  Std deviation: {cv_scores.std():.4f}")
print(f"  95% confidence interval: [{cv_scores.mean() - 1.96*cv_scores.std():.4f}, {cv_scores.mean() + 1.96*cv_scores.std():.4f}]")

print(f"\n✓ Cross-validation completed!")
print(f"  Model shows {'consistent' if cv_scores.std() < 0.01 else 'variable'} performance across folds")

## 9. Model Summary and Next Steps

Summary of model performance and recommendations for improvement.

In [ ]:
print("="*70)
print("MODEL SUMMARY")
print("="*70)

print(f"\n📊 DATASET:")
print(f"  Total records: {len(df_clean):,}")
print(f"  Training set: {len(X_train):,}")
print(f"  Test set: {len(X_test):,}")
print(f"  Features: {len(feature_cols)}")

print(f"\n🎯 MODEL PERFORMANCE:")
print(f"  Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Test F1-Score: {f1_score(y_test, y_test_pred):.4f}")
print(f"  Test ROC-AUC: {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"  Cross-Val Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n🔑 TOP 5 PREDICTIVE FEATURES:")
for idx, (i, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    direction = "increases" if row['Coefficient'] > 0 else "decreases"
    print(f"  {idx}. {row['Feature']} ({direction} win probability)")

print(f"\n💡 NEXT STEPS FOR IMPROVEMENT:")
print(f"  1. Try ensemble methods (Random Forest, XGBoost)")
print(f"  2. Feature engineering (create interaction features)")
print(f"  3. Hyperparameter tuning using GridSearchCV")
print(f"  4. Try different feature selection techniques")
print(f"  5. Consider temporal features (team momentum, recent performance)")
print(f"  6. Add opponent strength features (head-to-head stats)")

print("\n" + "="*70)
print("✓ Baseline model successfully created!")
print("  This model provides a solid foundation for predicting NCAA basketball game outcomes.")
print("="*70)